# Multivariate Linear Regression

## Overview
This notebook demonstrates **Multivariate Linear Regression**, which predicts a target variable using multiple independent features.

**Formula:** `y = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ`

Where:
- `y` = predicted value (dependent variable)
- `β₀` = intercept (bias term)
- `β₁, β₂, ..., βₙ` = coefficients (weights) for each feature
- `x₁, x₂, ..., xₙ` = independent variables (features)

In [ ]:
# Import required libraries
import pandas as pd              # For data manipulation and analysis
import numpy as np               # For numerical operations
import matplotlib.pyplot as plt  # For data visualization
from sklearn import linear_model # For linear regression model
from sklearn.metrics import r2_score, mean_squared_error  # For model evaluation

---
## Example 1: Home Price Prediction

**Objective:** Predict house prices based on multiple features (area, bedrooms, age)

**Independent Variables (X):**
- Area (square feet)
- Number of bedrooms
- Age of the house (years)

**Dependent Variable (y):** Price ($)

In [ ]:
# Step 1: Load the home prices dataset
df = pd.read_csv('homeprices (1).csv')

# Display the first 5 rows to understand the data structure
df.head()

In [ ]:
# Step 2: Handle missing values (Data Preprocessing)
# Missing values can cause errors in model training
# Strategy: Fill missing 'bedrooms' values with the median
# Why median? It's less affected by outliers than the mean

df["bedrooms"] = df["bedrooms"].fillna(df["bedrooms"].median())

# Verify that missing values are filled
df.head()

In [ ]:
# Step 3: Create and train the Linear Regression model
# Initialize the model
reg = linear_model.LinearRegression()

# Train the model using:
# X = features (area, bedrooms, age)
# y = target (price)
reg.fit(df[['area', 'bedrooms', 'age']], df['price'])

print("✓ Model training completed!")

In [ ]:
# Step 4: Extract model coefficients (weights)
# Coefficients show how much each feature impacts the price

print("Coefficients (β₁, β₂, β₃):", reg.coef_)
print("\nInterpretation:")
print(f"- Area coefficient: ${reg.coef_[0]:.2f} → For every 1 sq ft increase, price increases by ${reg.coef_[0]:.2f}")
print(f"- Bedrooms coefficient: ${reg.coef_[1]:.2f} → For every additional bedroom, price increases by ${reg.coef_[1]:.2f}")
print(f"- Age coefficient: ${reg.coef_[2]:.2f} → For every year older, price decreases by ${abs(reg.coef_[2]):.2f}")

In [ ]:
# Step 5: Get the intercept (β₀)
# The intercept is the base price when all features are zero

print(f"Intercept (β₀): ${reg.intercept_:.2f}")
print("\nComplete Equation:")
print(f"Price = {reg.coef_[0]:.2f} × Area + {reg.coef_[1]:.2f} × Bedrooms + {reg.coef_[2]:.2f} × Age + {reg.intercept_:.2f}")

In [ ]:
# Step 6: Make predictions for new houses
# Example: Predict price for a house with 3000 sq ft, 3 bedrooms, 40 years old

prediction = reg.predict([[3000, 3, 40]])
print(f"Predicted price for house (3000 sq ft, 3 bedrooms, 40 years): ${prediction[0]:,.2f}")

In [ ]:
# Step 7: Verify prediction with manual calculation
# This helps us understand how the model makes predictions

manual_calculation = (reg.coef_[0] * 3000 +      # Area contribution
                     reg.coef_[1] * 3 +          # Bedrooms contribution
                     reg.coef_[2] * 40 +         # Age contribution
                     reg.intercept_)             # Base price

print(f"Manual calculation: ${manual_calculation:,.2f}")
print(f"Model prediction:   ${prediction[0]:,.2f}")
print(f"✓ Both values match! The model works correctly.")

In [ ]:
# Step 8: Make another prediction
# Example: 2500 sq ft, 4 bedrooms, 5 years old

prediction2 = reg.predict([[2500, 4, 5]])
print(f"Predicted price for house (2500 sq ft, 4 bedrooms, 5 years): ${prediction2[0]:,.2f}")

---
## Example 2: Salary Prediction (Hiring Dataset)

**Objective:** Predict candidate salary based on experience, test scores, and interview performance

**Challenges:**
1. Missing values in multiple columns
2. Text data ("five", "ten") instead of numbers in experience column

**Independent Variables (X):**
- Experience (years)
- Test score (out of 10)
- Interview score (out of 10)

**Dependent Variable (y):** Salary ($)

In [ ]:
# Step 1: Load the hiring dataset
df_hiring = pd.read_csv(r"hiring.csv")

print("Original Dataset:")
print("Notice: 'experience' has text values like 'five', 'ten' instead of numbers")
print("Also: Some NaN values need to be handled\n")
df_hiring

In [ ]:
# Step 2: Handle missing test scores
# Fill missing test_score values with the median
# This is a common approach for numerical missing data

df_hiring["test_score(out of 10)"] = df_hiring["test_score(out of 10)"].fillna(
    df_hiring["test_score(out of 10)"].median()
)

print("✓ Test scores missing values filled with median")
df_hiring

In [ ]:
# Step 3: Convert text numbers to numeric values
# Problem: Experience column has text like "five", "ten", "eleven"
# Solution: Use word2number library to convert text to numbers

from word2number import w2n

# Define a safe conversion function that handles NaN and conversion errors
def convert_to_number(val):
    """
    Convert text numbers (e.g., 'five', 'ten') to numeric values.
    
    Args:
        val: Input value (can be text, number, or NaN)
    
    Returns:
        Numeric value if conversion successful, otherwise original value
    """
    if pd.isna(val):  # Keep NaN as NaN
        return val
    try:
        return w2n.word_to_num(val)  # Convert text like 'five' → 5
    except:
        return val  # If conversion fails, return original value

# Apply the conversion function to the experience column
df_hiring["experience"] = df_hiring["experience"].apply(convert_to_number)

print("✓ Text numbers converted to numeric values")
df_hiring

In [ ]:
# Step 4: Fill remaining missing experience values
# After conversion, some experience values might still be NaN
# Fill them with the median of the numeric experience values

df_hiring["experience"] = df_hiring["experience"].fillna(
    df_hiring["experience"].median()
)

print("✓ All missing values handled")
print("\nFinal cleaned dataset:")
df_hiring

In [ ]:
# Step 5: Train the salary prediction model
# Create a new model for hiring data

reg_hiring = linear_model.LinearRegression()

# Features: experience, test_score, interview_score
# Target: salary
reg_hiring.fit(
    df_hiring[['experience', 'test_score(out of 10)', 'interview_score(out of 10)']], 
    df_hiring["salary($)"]
)

print("✓ Salary prediction model trained successfully!")

In [ ]:
# Step 6: Display model coefficients and interpret them

print("=" * 70)
print("MODEL COEFFICIENTS (Impact of each feature on salary)")
print("=" * 70)
print(f"\nCoefficients: {reg_hiring.coef_}")
print(f"\nInterpretation:")
print(f"- Experience: ${reg_hiring.coef_[0]:.2f} per year")
print(f"  → Each additional year of experience increases salary by ${reg_hiring.coef_[0]:.2f}")
print(f"\n- Test Score: ${reg_hiring.coef_[1]:.2f} per point")
print(f"  → Each additional test point increases salary by ${reg_hiring.coef_[1]:.2f}")
print(f"\n- Interview Score: ${reg_hiring.coef_[2]:.2f} per point")
print(f"  → Each additional interview point increases salary by ${reg_hiring.coef_[2]:.2f}")
print(f"\nIntercept: ${reg_hiring.intercept_:.2f}")
print("  → Base salary when all features are zero")

In [ ]:
# Step 7: Make salary predictions for new candidates

# Example 1: Candidate with 2 years experience, test score 9, interview score 6
candidate1_salary = reg_hiring.predict([[2, 9, 6]])
print(f"Candidate 1 (2 yrs exp, test: 9, interview: 6)")
print(f"Predicted Salary: ${candidate1_salary[0]:,.2f}")

print("\n" + "-" * 50 + "\n")

# Example 2: Candidate with 12 years experience, test score 10, interview score 10
candidate2_salary = reg_hiring.predict([[12, 10, 10]])
print(f"Candidate 2 (12 yrs exp, test: 10, interview: 10)")
print(f"Predicted Salary: ${candidate2_salary[0]:,.2f}")

In [ ]:
# Step 8: Evaluate model performance
# R² Score measures how well the model fits the data (0 to 1, higher is better)

y_pred = reg_hiring.predict(
    df_hiring[['experience', 'test_score(out of 10)', 'interview_score(out of 10)']]
)
r2 = r2_score(df_hiring["salary($)"], y_pred)

print("=" * 70)
print("MODEL PERFORMANCE")
print("=" * 70)
print(f"R² Score: {r2:.4f}")
print(f"Accuracy: {r2 * 100:.2f}%")
print("\nInterpretation:")
if r2 > 0.9:
    print("✓ Excellent fit! The model explains more than 90% of salary variation.")
elif r2 > 0.7:
    print("✓ Good fit! The model explains more than 70% of salary variation.")
else:
    print("⚠ Moderate fit. Consider adding more features or more data.")

---
## Key Learnings

### 1. Multivariate Linear Regression Formula
```
y = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ
```

### 2. Data Preprocessing Steps
- **Handle missing values:** Use median/mean for numerical data
- **Convert data types:** Ensure all features are numeric
- **Text to numbers:** Use libraries like word2number for text conversion

### 3. Model Interpretation
- **Coefficients (β):** Show the impact of each feature on the target
- **Positive coefficient:** Feature increases with target
- **Negative coefficient:** Feature decreases with target
- **Intercept (β₀):** Base value when all features are zero

### 4. Model Evaluation
- **R² Score:** Measures model fit (0 to 1)
  - 1.0 = Perfect fit
  - 0.9+ = Excellent
  - 0.7-0.9 = Good
  - < 0.7 = Needs improvement

### 5. When to Use Multivariate Linear Regression
- ✓ Predicting continuous numerical values
- ✓ When you have multiple independent variables
- ✓ When relationships are approximately linear
- ✗ Not suitable for classification tasks (use Logistic Regression instead)